Não temos valores nulos

Temos linhas duplicatas, mas vamos considerar que são pacientes distintos

As seguintes colunas vamos fazer one hot encoding: Gender, family_history, FAVC, CAEC, SMOKE, SCC, CALC, MTRANS

# Bibliotecas

In [209]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Import da base original

In [ ]:
CSV_PATH = "Obesity.csv"  # ajuste o nome do arquivo, se necessário

df = pd.read_csv(CSV_PATH)

# Padronizar nomes das colunas
df.columns = df.columns.str.strip()

df.drop_duplicates(inplace=True)

# Padronizar valores textuais
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

# rename_map = {
#     "Gender": "genero",
#     "Age": "idade",
#     "Height": "altura",
#     "Weight": "peso",
#     "family_history": "flag_historico_familiar",
#     "FAVC": "flag_consumo_frequente_alimentos_caloricos",
#     "FCVC": "frequencia_consumo_vegetais", # INT
#     "NCP": "numero_refeicoes_principais_dia", # INT
#     "CAEC": "consume_alimentos_entre_refeicoes", # CAT
#     "SMOKE": "flag_fumante",
#     "CH2O": "numero_consumo_agua",
#     "SCC": "flag_monitora_calorias",
#     "FAF": "frequencia_atividade_fisica",
#     "TUE": "tempo_uso_dispositivos_eletronicos",
#     "CALC": "consumo_bebida_alcoolica",
#     "MTRANS": "meio_de_transporte",
# }

# df = df.rename(columns=rename_map)

print("Dimensão original da base:")
print(df.shape)

display(df.head())

Dimensão original da base:
(2087, 17)


,Gender,Age,Height,Weight,family_history,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,Obesity
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


# Definindo a target

In [212]:
# 3. DEFINIÇÃO DA TARGET
target = "Obesity"

target_map = {
    "Insufficient_Weight": 0,
    "Normal_Weight": 1,
    "Overweight_Level_I": 2,
    "Overweight_Level_II": 3,
    "Obesity_Type_I": 4,
    "Obesity_Type_II": 5,
    "Obesity_Type_III": 6
}

# Criar uma cópia da target original para consulta, se quiser
df["target_original"] = df[target]

# Aplicar o mapeamento
df["target_numerico"] = df[target].map(target_map)

# Checar se alguma classe ficou sem mapeamento
if df[target].isnull().sum() > 0:
    print("Atenção: existem classes da target que não foram mapeadas.")
    print(df.loc[df["target_numerico"].isnull(), "target_original"].unique())
else:
    print("Mapeamento da target realizado com sucesso.")

# Garantir tipo inteiro
df["target_numerico"] = df["target_numerico"].astype(int)

# display(
#     df[["target_original", "target_numerico"]]
#     .drop_duplicates()
#     .sort_values("target_numerico")
# )

# Distribuição da target

X = df.drop(columns=[target,"target_original","target_numerico"])
y = df["target_numerico"]

print("Dimensão de X:")
print(X.shape)

print("\nDimensão de y:")
print(y.shape)

print("\nDistribuição da target:")
display(y.value_counts(normalize=True).mul(100).round(2))

Mapeamento da target realizado com sucesso.
Dimensão de X:
(2087, 16)

Dimensão de y:
(2087,)

Distribuição da target:


target_numerico
4    16.82
6    15.52
5    14.23
3    13.90
1    13.51
2    13.22
0    12.79
Name: proportion, dtype: float64

# Separando os conjuntos dos dados: Treino, Validação e Teste

In [213]:
# Estratégia:
# - 70% treino
# - 15% validação
# - 15% teste

RANDOM_STATE = 42

# Primeiro split: separa treino de base temporária
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

# Segundo split: divide a base temporária em validação e teste
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print(f"Treino:    X_train = {X_train.shape} | y_train = {y_train.shape}")
print(f"Validação: X_val   = {X_val.shape} | y_val   = {y_val.shape}")
print(f"Teste:     X_test  = {X_test.shape} | y_test  = {y_test.shape}")


Treino:    X_train = (1460, 16) | y_train = (1460,)
Validação: X_val   = (313, 16) | y_val   = (313,)
Teste:     X_test  = (314, 16) | y_test  = (314,)


## Vamos salvar as bases para garantir confiança do modelo

In [ ]:
# ## ATENÇÃO: VAMOS DEIXAR O ULTIMO TRECHO COMENTADO PARA GARANTIR QUE NÃO VAMOS SALVAR EM CIMA DAS BASES CRIADAS

# import os

# output_dir = "bases_preparadas"
# os.makedirs(output_dir, exist_ok=True)

# X_train.to_csv(f"{output_dir}/X_train.csv", index=False)
# X_val.to_csv(f"{output_dir}/X_val.csv", index=False)
# X_test.to_csv(f"{output_dir}/X_test.csv", index=False)

# y_train.to_csv(f"{output_dir}/y_train.csv", index=False)
# y_val.to_csv(f"{output_dir}/y_val.csv", index=False)
# y_test.to_csv(f"{output_dir}/y_test.csv", index=False)

# print("Bases X e y salvas com sucesso.")

Bases X e y salvas com sucesso.


# Pré-Processamento da base de Treino

In [198]:
output_dir = "bases_preparadas"

X_train = pd.read_csv(f"{output_dir}/X_train.csv")
y_train = pd.read_csv(f"{output_dir}/y_train.csv")

# df_treino = pd.concat([X_train,y_train],axis=1)

## Garantindo indicações do dicionário

In [199]:
## Conforme orientado no dicionário, vamos garantir que os valores são inteiros

cols_to_round = ["FCVC", "NCP", "CH2O", "FAF", "TUE"]

for col in cols_to_round:
    if col in X_train.columns:
        X_train[col] = X_train[col].round().astype(int)

# print("Valores únicos após arredondamento:")

# for col in cols_to_round:
#     if col in X_train.columns:
#         print(f"{col}:", sorted(X_train[col].unique()))

## Nulos e duplicatas

In [200]:
# print("Quantidade de nulos por coluna:")
# display(X.isnull().sum())

# print("\nQuantidade de linhas duplicadas:")
# print(X_train.duplicated().sum())

## Criando features

In [201]:
## Criando Features

def create_features(df):
    df = df.copy()

    df["flag_female"] = df["Gender"].map({"Female": 1, "Male": 0})
    df = df.drop(columns=["Gender"])

    df["flag_family_history"] = df["family_history"].map({"yes": 1, "no": 0})
    df = df.drop(columns=["family_history"])

    df["flag_FAVC"] = df["FAVC"].map({"yes": 1, "no": 0})
    df = df.drop(columns=["FAVC"])

    df["flag_SMOKE"] = df["SMOKE"].map({"yes": 1, "no": 0})
    df = df.drop(columns=["SMOKE"])

    df["flag_SCC"] = df["SCC"].map({"yes": 1, "no": 0})
    df = df.drop(columns=["SCC"])

    df["flag_active_transport"] = (df["MTRANS"].isin(["Bike", "Walking"])).astype(int)

    mapa_frequencia = {
        "no": 0,
        "Sometimes": 1,
        "Frequently": 2,
        "Always": 3
    }

    df["CAEC_freq"] = df["CAEC"].map(mapa_frequencia)
    df["CALC_freq"] = df["CALC"].map(mapa_frequencia)

    df = df.drop(columns=["CAEC", "CALC"])

    colunas_categoricas = ["MTRANS"]

    df = pd.get_dummies(df, columns=colunas_categoricas, dtype=int)

    # IMC
    df["BMI"] = df["Weight"] / (df["Height"] ** 2)

    # Interação IMC x idade
    df["BMI_x_Age"] = df["BMI"] * df["Age"]

    # Score de sedentarismo
    df["sedentary_score"] = (
        (3-df["FAF"]) +
        df["TUE"] +
        df["MTRANS_Automobile"] +
        df["MTRANS_Motorbike"] +
        df["MTRANS_Public_Transportation"] 
    )

    # Score de proteção por atividade
    df["activity_protection_score"] = (
        df["FAF"] +
        df["flag_active_transport"]
    )

    return df

X_train = create_features(X_train)

## Filtragens

In [202]:
# Thresholds
null_threshold = 0.90
dominance_threshold = 0.99
var_threshold = 1e-5

# Filtragem calculada apenas na base de treino

cols_high_null = X_train.columns[
    X_train.isna().mean() > null_threshold
].tolist()

cols_constant = X_train.columns[
    X_train.nunique(dropna=False) <= 1
].tolist()

cols_high_dominance = [
    col for col in X_train.columns
    if X_train[col].value_counts(normalize=True, dropna=False).iloc[0] > dominance_threshold
]

cols_low_variance = X_train.var()[X_train.var() < var_threshold].index.tolist()

# Consolidar colunas para remover

cols_to_drop = list(set(
    cols_high_null +
    cols_constant +
    cols_high_dominance +
    cols_low_variance
))

print("Colunas removidas:", cols_to_drop)
print("Quantidade:", len(cols_to_drop))

# Aplicar filtro na base de treino

X_train = X_train.drop(columns=cols_to_drop)

Colunas removidas: ['MTRANS_Bike', 'MTRANS_Motorbike']
Quantidade: 2


## Filtrando correlações

In [203]:
from scipy.stats import chi2_contingency

X_corr = X_train.copy()
y_corr = y_train.copy()

if isinstance(y_corr, pd.DataFrame):
    y_corr = y_corr.iloc[:, 0]

y_corr = y_corr.squeeze()

binary_cols = [
    col for col in X_corr.columns
    if set(X_corr[col].dropna().unique()).issubset({0, 1})
]

numeric_cols = [
    col for col in X_corr.columns
    if col not in binary_cols
]

print("Numéricas:", numeric_cols)
print("Binárias:", binary_cols)

Numéricas: ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE', 'CAEC_freq', 'CALC_freq', 'BMI', 'BMI_x_Age', 'sedentary_score', 'activity_protection_score']
Binárias: ['flag_female', 'flag_family_history', 'flag_FAVC', 'flag_SMOKE', 'flag_SCC', 'flag_active_transport', 'MTRANS_Automobile', 'MTRANS_Public_Transportation', 'MTRANS_Walking']


### Variáveis numéricas

#### Entre variáveis

In [204]:
spearman_num = X_corr[numeric_cols].corr(method="spearman")

threshold = 0.85

corr_pairs_num = (
    spearman_num.abs()
    .where(np.triu(np.ones(spearman_num.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)

corr_pairs_num.columns = ["var_1", "var_2", "spearman_abs"]

high_corr_num = corr_pairs_num.query("spearman_abs >= @threshold") \
                              .sort_values("spearman_abs", ascending=False)

display(high_corr_num)

,var_1,var_2,spearman_abs
97,FAF,activity_protection_score,0.988536
38,Weight,BMI,0.926601


#### Entre variáveis e Target

In [205]:
spearman_target = (
    X_corr[numeric_cols]
    .corrwith(y_corr, method="spearman")
    .abs()
    .sort_values(ascending=False)
)

display(spearman_target)

BMI                          0.988048
Weight                       0.917034
BMI_x_Age                    0.822126
Age                          0.399612
CAEC_freq                    0.385044
activity_protection_score    0.215196
FCVC                         0.208325
FAF                          0.199600
CALC_freq                    0.172018
CH2O                         0.138227
sedentary_score              0.129792
Height                       0.125777
TUE                          0.077615
NCP                          0.026869
dtype: float64

### Variáveis binárias

#### Entre variáveis

In [206]:
def cramers_v(x, y):
    table = pd.crosstab(x.squeeze(), y.squeeze())
    chi2 = chi2_contingency(table)[0]
    n = table.sum().sum()
    r, k = table.shape
    
    if min(r - 1, k - 1) == 0:
        return np.nan
    
    return np.sqrt((chi2 / n) / min(k - 1, r - 1))

cramer_bin = pd.DataFrame(index=binary_cols, columns=binary_cols, dtype=float)

for col1 in binary_cols:
    for col2 in binary_cols:
        cramer_bin.loc[col1, col2] = cramers_v(X_corr[col1], X_corr[col2])

# display(cramer_bin)

corr_pairs_bin = (
    cramer_bin
    .where(np.triu(np.ones(cramer_bin.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)

corr_pairs_bin.columns = ["var_1", "var_2", "cramers_v"]

high_corr_bin = corr_pairs_bin.query("cramers_v >= @threshold") \
                              .sort_values("cramers_v", ascending=False)

display(high_corr_bin)

,var_1,var_2,cramers_v
61,MTRANS_Automobile,MTRANS_Public_Transportation,0.914576
53,flag_active_transport,MTRANS_Walking,0.903641


#### Entre variáveis e Target

In [207]:
cramer_target = pd.Series({
    col: cramers_v(X_corr[col], y_corr)
    for col in binary_cols
}).sort_values(ascending=False)

display(cramer_target)

flag_female                     0.565093
flag_family_history             0.561812
flag_FAVC                       0.312758
MTRANS_Automobile               0.278115
MTRANS_Public_Transportation    0.275500
flag_SCC                        0.251858
flag_active_transport           0.173996
MTRANS_Walking                  0.157859
flag_SMOKE                      0.130932
dtype: float64

### Colunas sendo removidas pós-filtragem

In [208]:
cols_to_drop_spearman = ["FAF", "Weight"]
cols_to_drop_cramer = []
cols_to_drop_spearman_target = ["TUE", "NCP"]
cols_to_drop_cramer_target = []
X_train = X_train.drop(columns=cols_to_drop_spearman+cols_to_drop_cramer+cols_to_drop_spearman_target+cols_to_drop_cramer_target)